In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
data_dir = "/content/drive/MyDrive"

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler

import matplotlib.pyplot as plt

from PIL import Image
import torchvision
from torchvision import transforms, datasets

from torch.utils.data import (
    DataLoader,
    random_split
)

In [3]:
import shutil

if not os.path.exists("/content/dataset_split"):

    shutil.copytree(
        "/content/drive/MyDrive/dataset_split",
        "/content/dataset_split"
    )

print("Dataset skopiowany lokalnie")

Dataset skopiowany lokalnie


In [20]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])



train_dir = "/content/dataset_split/train"
val_dir = "/content/dataset_split/val"
test_dir = "/content/dataset_split/test"

train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root=val_dir,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=test_dir,
    transform=transform
)

# dataloadery
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

# klasy
class_names = train_dataset.classes

print("Klasy:")
print(class_names)

print(f"\nTrain images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")
print(f"Test images: {len(test_dataset)}")

Klasy:
['APSK128', 'APSK16', 'APSK32', 'APSK64', 'ASK4', 'ASK8', 'BPSK', 'GMSK', 'PSK16', 'PSK32', 'PSK8', 'QAM128', 'QAM16', 'QAM256', 'QAM32', 'QAM64', 'QPSK']

Train images: 18370
Val images: 3923
Test images: 3923


# **ResNet50**

In [21]:
device = torch.device("cuda")

from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

model.fc = nn.Linear(
    model.fc.in_features,
    len(train_dataset.classes)
)

model = model.to(device)

#3 stage

In [23]:
for param in model.parameters():
    param.requires_grad = False

#for param in model.layer4.parameters():
    #param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

for param in model.layer4[2].parameters():
    param.requires_grad = True

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
#resnet50  stage3
optimizer = optim.Adam( model.fc.parameters(),lr=6e-5)

#resnet50
#optimizer = optim.Adam([
    #{'params': model.layer4.parameters(), 'lr': 3e-6},
    #{'params': model.fc.parameters(),     'lr': 6e-5}
#])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

In [ ]:
print(type(model))
print(model)

<class 'torchvision.models.resnet.ResNet'>
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv

In [24]:
num_epochs = 20
best_val_accuracy = 0.0

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    if epoch == 5:

        print("Stage 2")

        for param in model.layer4[2].parameters():
            param.requires_grad = True

        optimizer = optim.Adam([
            {'params': model.layer4[2].parameters(), 'lr': 1e-5},
            {'params': model.fc.parameters(), 'lr': 6e-5}
        ])
    if epoch == 12:
      print("Stage 3")

      for param in model.layer4.parameters():
          param.requires_grad = True

      optimizer = optim.Adam([
            {'params': model.layer4.parameters(), 'lr': 2e-6},
            {'params': model.fc.parameters(), 'lr': 2e-5}
      ])
    print(f"\nEpoka {epoch+1}/{num_epochs}")



    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)



    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)



    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f} | "
        f"Val acc:   {val_accuracy:.4f}"
    )



    torch.save(
        model.state_dict(),
        "/content/drive/MyDrive/runs_resnet50_stage3/model_current_resnet50.pth"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/runs_resnet50_stage3/model_best_resnet50.pth"
        )

        print("Zapisano najlepszy model")

    scheduler.step(val_accuracy)

print(f"Best accuracy: {best_val_accuracy:.4f}")


Epoka 1/20
Train loss: 2.2794 | Train acc: 0.3883
Val loss:   2.0364 | Val acc:   0.4871
Zapisano najlepszy model

Epoka 2/20
Train loss: 1.6123 | Train acc: 0.6137
Val loss:   1.6245 | Val acc:   0.5570
Zapisano najlepszy model

Epoka 3/20
Train loss: 1.3165 | Train acc: 0.6715
Val loss:   1.4699 | Val acc:   0.5690
Zapisano najlepszy model

Epoka 4/20
Train loss: 1.1482 | Train acc: 0.7032
Val loss:   1.3282 | Val acc:   0.5886
Zapisano najlepszy model

Epoka 5/20
Train loss: 1.0352 | Train acc: 0.7302
Val loss:   1.2562 | Val acc:   0.6138
Zapisano najlepszy model
Stage 2

Epoka 6/20
Train loss: 0.7551 | Train acc: 0.7847
Val loss:   0.9010 | Val acc:   0.6844
Zapisano najlepszy model

Epoka 7/20
Train loss: 0.5275 | Train acc: 0.8342
Val loss:   0.7067 | Val acc:   0.7527
Zapisano najlepszy model

Epoka 8/20
Train loss: 0.4362 | Train acc: 0.8573
Val loss:   0.7282 | Val acc:   0.7545
Zapisano najlepszy model

Epoka 9/20
Train loss: 0.3846 | Train acc: 0.8698
Val loss:   0.6455 | 

In [ ]:
# @title
num_epochs = 20
best_val_accuracy = 0.0

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    if epoch == 8:

        print("Odblokowuję layer4")

        for param in model.layer4.parameters():
            param.requires_grad = True

        optimizer = optim.Adam([
            {'params': model.layer4.parameters(), 'lr': 3e-5},
            {'params': model.fc.parameters(),     'lr': 6e-5}
        ])

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2
    )

    print(f"\nEpoka {epoch+1}/{num_epochs}")



    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)



    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)



    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f} | "
        f"Val acc:   {val_accuracy:.4f}"
    )



    torch.save(
        model.state_dict(),
        "/content/drive/MyDrive/runs_resnet50_stage2/model_current_resnet50.pth"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/runs_resnet50_stage2/model_best_resnet50.pth"
        )

        print("Zapisano najlepszy model")

    scheduler.step(val_accuracy)

print(f"Best accuracy: {best_val_accuracy:.4f}")


Epoka 1/20
Train loss: 2.2706 | Train acc: 0.4065
Val loss:   1.9992 | Val acc:   0.4871
Zapisano najlepszy model

Epoka 2/20
Train loss: 1.6099 | Train acc: 0.6210
Val loss:   1.6392 | Val acc:   0.5391
Zapisano najlepszy model

Epoka 3/20
Train loss: 1.3145 | Train acc: 0.6758
Val loss:   1.4572 | Val acc:   0.5623
Zapisano najlepszy model

Epoka 4/20
Train loss: 1.1478 | Train acc: 0.7100
Val loss:   1.3243 | Val acc:   0.6069
Zapisano najlepszy model

Epoka 5/20
Train loss: 1.0364 | Train acc: 0.7276
Val loss:   1.2260 | Val acc:   0.6102
Zapisano najlepszy model

Epoka 6/20
Train loss: 0.9561 | Train acc: 0.7475
Val loss:   1.1910 | Val acc:   0.6082

Epoka 7/20
Train loss: 0.8906 | Train acc: 0.7602
Val loss:   1.1554 | Val acc:   0.6373
Zapisano najlepszy model

Epoka 8/20
Train loss: 0.8444 | Train acc: 0.7666
Val loss:   1.0941 | Val acc:   0.6424
Zapisano najlepszy model
Odblokowuję layer4

Epoka 9/20
Train loss: 0.4131 | Train acc: 0.8519
Val loss:   0.4190 | Val acc:   0.8

In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/runs_resnet50_stage3/model_best_resnet50.pth"
    )
)

model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

In [26]:
runs_dir = "/content/drive/MyDrive/runs_resnet50_stage3"
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(true_labels, pred_labels)

precision = precision_score(
    true_labels,
    pred_labels,
    average='macro'
)

recall = recall_score(
    true_labels,
    pred_labels,
    average='macro'
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average='macro'
)

print("\nWyniki modelu")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

metrics_file = os.path.join(
    runs_dir,
    "test_metrics.txt"
)

with open(metrics_file, "w") as f:

    f.write("Wyniki modelu\n")
    f.write(f"Accuracy : {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall   : {recall:.4f}\n")
    f.write(f"F1-score : {f1:.4f}\n")

print(f"\nZapisano do: {metrics_file}")


Wyniki modelu
Accuracy : 0.8542
Precision: 0.8801
Recall   : 0.8499
F1-score : 0.8539

Zapisano do: /content/drive/MyDrive/runs_resnet50_stage3/test_metrics.txt


In [27]:
import seaborn as sns

from sklearn.metrics import confusion_matrix
runs_dir = "/content/drive/MyDrive/runs_resnet50_stage3"

os.makedirs(runs_dir, exist_ok=True)

# loss plot

plt.figure(figsize=(10,5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')

plt.title("Loss")
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "loss_plot.png"),
    bbox_inches='tight'
)

plt.close()

# accuracy plot

plt.figure(figsize=(10,5))

plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Val Accuracy')

plt.title("Accuracy")
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "accuracy_plot.png"),
    bbox_inches='tight'
)

plt.close()

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    true_labels,
    pred_labels
)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_dataset.classes,
    yticklabels=test_dataset.classes
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.savefig(
    os.path.join(
        runs_dir,
        "test_confusion_matrix.png"
    ),
    bbox_inches="tight"
)

plt.close()

print("\nZapisano:")
print("- model_best.pth")
print("- model_current.pth")
print("- loss_plot.png")
print("- accuracy_plot.png")
print("- confusion_matrix.png")


Zapisano:
- model_best.pth
- model_current.pth
- loss_plot.png
- accuracy_plot.png
- confusion_matrix.png
